In [0]:
%run "../common/config"


In [0]:
%run "../common/io_utils"

In [0]:


# COMMAND ----------
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta

bronze_table = f"{BRONZE}.index_price_raw"
failed_table = f"{BRONZE}.failed_ingestion_index"

index_list = [row.Symbol for row in spark.table(f"{BRONZE}.index_master_raw").select("Symbol").collect()]
max_date_dict = get_watermark_dict(spark, WATERMARK_TABLE, "index_price")
end_date = datetime.today().strftime("%Y-%m-%d")

all_data, failed_symbols = [], []

for s in index_list:
    last_date = max_date_dict.get(s)
    start_date = (
        (pd.to_datetime(last_date) + timedelta(days=1)).strftime("%Y-%m-%d")
        if last_date else DEFAULT_BACKFILL_START
    )
    try:
        data = yf.download(s, start=start_date, end=end_date, progress=False)
        if data.empty:
            failed_symbols.append(s)
            continue
        df_symbol = clean_price_frame(data.reset_index(), s)
        if "close" in df_symbol.columns:
            df_symbol = df_symbol.dropna(subset=["close"])
        if not df_symbol.empty:
            all_data.append(df_symbol)
    except Exception:
        failed_symbols.append(s)

if all_data:
    combined_df = pd.concat(all_data)
    combined_df = combined_df.loc[:, ~combined_df.columns.duplicated()]
    spark.createDataFrame(combined_df, schema=PRICE_SCHEMA) \
        .write.format("delta").mode("append") \
        .partitionBy("symbol").saveAsTable(bronze_table)

    batch_max_dates = combined_df.groupby("symbol")["date"].max().dt.date.to_dict()
    update_watermark(spark, WATERMARK_TABLE, "index_price", batch_max_dates)

if failed_symbols:
    spark.createDataFrame([(s, datetime.now()) for s in failed_symbols], ["symbol", "failed_at"]) \
        .write.format("delta").mode("append").saveAsTable(failed_table)

print(f"Done. {len(all_data)} symbols updated, {len(failed_symbols)} failed.")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ^NSENEXT"}}}
$^NSENEXT: possibly delisted; no timezone found

1 Failed download:
['^NSENEXT']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ^CNXMIDCAP"}}}
$^CNXMIDCAP: possibly delisted; no timezone found

1 Failed download:
['^CNXMIDCAP']: possibly delisted; no timezone found
$^CNXFINANCE: possibly delisted; no timezone found

1 Failed download:
['^CNXFINANCE']: possibly delisted; no timezone found


Done. 17 symbols updated, 3 failed.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
